# Wasserstein risk index (distribution shift) for futures.

This notebook demonstrates a small end-to-end research pipeline:

- Load continuous daily futures prices via PostgreSQL
- Compute daily log returns
- Build the Wasserstein distribution-shift index $W_t$ from cross-sectional returns
- Build a core panel with realized vol + rolling largest eigenvalue
- Run a simple HAC regression

In [ ]:
# 0) Setup & Imports
%load_ext autoreload
%autoreload 2

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from IPython.display import display

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from algogators_wrisk import config, features, analysis

In [ ]:
# 1) Check configuration
print(f"Universe size: {len(config.UNIVERSE)}")
print(f"Date range: {config.START_DATE} \u2192 {config.END_DATE}")
print(f"RV windows (past/future): {config.RV_PAST_WINDOW}/{config.RV_FUTURE_WINDOW}")
print(f"Lambda1 window: {config.LAMBDA1_WINDOW}")
print(f"Event quantile: {config.W_EVENT_QUANTILE}")
print(f"Exposure on event: {config.EXPOSURE_ON_EVENT}")
print(f"HAC lags: {config.HAC_LAGS}")

In [ ]:
# 2) Fetch Data from PostgreSQL using SQLAlchemy
print("--- Connecting & Loading Data ---")
conn_str = f"postgresql+psycopg://{os.environ['DB_USER']}:{os.environ['DB_PASSWORD']}@{os.environ['DB_HOST']}:5432/{os.environ['DB_NAME']}"
engine = create_engine(conn_str)

def fetch_data(symbols, start, end):
    price_series = []
    for symbol in symbols:
        query = f"SELECT time, close FROM {config.DB_SCHEMA}.{config.PRICES_TABLE} WHERE symbol = '{symbol}' AND time BETWEEN '{start}' AND '{end}' ORDER BY time ASC"
        df = pd.read_sql(query, engine)
        if not df.empty:
            df['time'] = pd.to_datetime(df['time']).dt.floor('D')
            s = df.set_index('time')['close'].rename(symbol)
            price_series.append(s[~s.index.duplicated(keep='last')])
    
    if not price_series:
        raise ValueError("No data found. Verify your config.py symbols and table names.")
        
    return pd.concat(price_series, axis=1).sort_index().ffill().dropna()

all_data = fetch_data(config.UNIVERSE, config.START_DATE, config.END_DATE)
print(f"\u2705 Success! Data loaded for {all_data.shape[1]} symbols:\n{list(all_data.columns)}")
display(all_data.head())

In [ ]:
# 3) Separate Universe prices from Macro Treasuries & Compute Returns
prices = all_data.drop(columns=['ZT', 'ZF'], errors='ignore')
df_macro = all_data[['ZT', 'ZF']].rename(columns={'ZT': 'treas_2y_close', 'ZF': 'treas_5y_close'}) if 'ZT' in all_data.columns else pd.DataFrame()

returns = np.log(prices).diff().dropna()
display(returns.head())

In [ ]:
# 4) Build the Core Panel (Generates Wasserstein 'W' index and Lambda1)
panel = analysis.build_core_panel(
    returns,
    rv_past_window=config.RV_PAST_WINDOW,
    rv_future_window=config.RV_FUTURE_WINDOW,
    lambda1_window=config.LAMBDA1_WINDOW
)

# Join Macro data if available
if not df_macro.empty:
    panel = panel.join(df_macro, how='inner').dropna()
    
display(panel.tail())

In [ ]:
# 5) Run the RV Regression model
results = analysis.run_rv_regression(
    panel,
    hac_lags=config.HAC_LAGS
)

print("\n--- Wasserstein Risk Index Regression ---")
print(results.summary())

---
## Exploratory Visualizations

In [ ]:
sns.set_theme(style="whitegrid")

# Plot Cumulative Returns of the Universe
cum_returns = (1 + returns).cumprod()
cum_returns.plot(figsize=(12, 6), colormap="tab10", alpha=0.8)
plt.title("Cumulative Returns (Continuous Futures)", fontsize=14)
plt.ylabel("Cumulative Return")
plt.xlabel("Date")
plt.tight_layout()
plt.show()

In [ ]:
# Plot Wasserstein Risk Index vs Realized Volatility
fig, ax1 = plt.subplots(figsize=(14, 6))

color = "tab:blue"
ax1.set_xlabel("Date")
ax1.set_ylabel("Wasserstein Risk Index (W)", color=color)
ax1.plot(panel.index, panel["W"], color=color, alpha=0.9, label="W Index")
ax1.tick_params(axis="y", labelcolor=color)

ax2 = ax1.twinx()  
color = "tab:red"
ax2.set_ylabel("Past Realized Volatility (rv_past)", color=color)  
ax2.plot(panel.index, panel["rv_past"], color=color, alpha=0.5, linestyle="--", label="RV Past")
ax2.tick_params(axis="y", labelcolor=color)

fig.tight_layout()  
plt.title("Distribution Shift (Wasserstein Risk) vs Realized Volatility", fontsize=14)
plt.show()